
# Roxy notebook example: User-defined motif descriptors via regex

This notebook is a **reference implementation example** for the **user-defined regex motif descriptor family** in Roxy.

Unlike predefined motif libraries, this family is designed to let the user define arbitrary regular-expression patterns and then convert them into descriptor tables.

This is especially useful when the user wants to track:

- custom biochemical patterns
- project-specific motifs
- candidate active-site signatures
- linker-like patterns
- repetitive local motifs
- sequence constraints derived from prior biological knowledge

## Covered outputs

This notebook implements:

- sequence cleaning
- user-defined regex motif dictionaries
- motif presence / absence
- motif counts
- motif density normalized by sequence length
- optional overlapping motif counting
- N-terminal and C-terminal motif occurrence
- first and last motif position
- normalized first and last motif positions
- span between first and last motif hit
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

import re
import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "uregex_1",
            "uregex_2",
            "uregex_3",
            "uregex_4",
            "uregex_5",
            "uregex_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,uregex_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,uregex_2,GGGGGGGGGGGGGGG,B
2,uregex_3,KRRKRRKRRKRRDDDDEE,A
3,uregex_4,ACDEFGHIKLMNPQRSTVWY,B
4,uregex_5,PPPPGSSSSSTTTTNNQQQ,A
5,uregex_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Example user-defined motif collection.
USER_REGEX_PATTERNS = {
    "basic_pair": r"KR|RK|KK|RR",
    "acidic_pair": r"DE|ED|DD|EE",
    "gly_run_2plus": r"G{2,}",
    "pro_run_2plus": r"P{2,}",
    "ser_thr_patch": r"[ST]{3,}",
    "charged_triplet": r"[KRHDE]{3,}",
    "aromatic_pair": r"[FWYH]{2,}",
    "amide_pair": r"[NQ]{2,}",
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def compile_pattern(pattern: str, overlapping: bool = False):
    if overlapping:
        return re.compile(f"(?=({pattern}))")
    return re.compile(pattern)


def find_matches(seq: str, pattern: str, overlapping: bool = False):
    seq = clean_sequence(seq)
    regex = compile_pattern(pattern, overlapping=overlapping)
    return list(regex.finditer(seq))


def match_count(seq: str, pattern: str, overlapping: bool = False) -> int:
    return len(find_matches(seq, pattern, overlapping=overlapping))


def match_presence(seq: str, pattern: str, overlapping: bool = False) -> int:
    return int(match_count(seq, pattern, overlapping=overlapping) > 0)


def match_density(seq: str, pattern: str, overlapping: bool = False) -> float:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.nan
    return match_count(seq, pattern, overlapping=overlapping) / len(seq)


def match_positions(seq: str, pattern: str, overlapping: bool = False):
    matches = find_matches(seq, pattern, overlapping=overlapping)
    positions = [m.start() + 1 for m in matches]
    return positions


def first_match_position(seq: str, pattern: str, overlapping: bool = False) -> float:
    pos = match_positions(seq, pattern, overlapping=overlapping)
    return float(pos[0]) if len(pos) > 0 else np.nan


def last_match_position(seq: str, pattern: str, overlapping: bool = False) -> float:
    pos = match_positions(seq, pattern, overlapping=overlapping)
    return float(pos[-1]) if len(pos) > 0 else np.nan


def normalized_position(pos: float, seq_len: int) -> float:
    if seq_len == 0 or np.isnan(pos):
        return np.nan
    return pos / seq_len


def match_span(seq: str, pattern: str, overlapping: bool = False) -> float:
    pos = match_positions(seq, pattern, overlapping=overlapping)
    if len(pos) < 2:
        return np.nan
    return float(pos[-1] - pos[0])


def terminal_presence(seq: str, pattern: str, side: str = "N", window: int = 10, overlapping: bool = False) -> int:
    seq = clean_sequence(seq)
    if side == "N":
        subseq = seq[:window]
    elif side == "C":
        subseq = seq[-window:]
    else:
        raise ValueError("side must be 'N' or 'C'")
    return match_presence(subseq, pattern, overlapping=overlapping)


## Core descriptor function

In [5]:

def regex_motif_descriptors(
    seq: str,
    pattern_dict: dict,
    overlapping: bool = False,
    include_terminal: bool = True,
    terminal_window: int = 10,
) -> dict:
    seq = clean_sequence(seq)
    seq_len = len(seq)

    out = {
        "uregex_length": seq_len,
        "uregex_valid_residue_count": seq_len,
        "uregex_overlapping_mode": int(overlapping),
    }

    for motif_name, pattern in pattern_dict.items():
        count = match_count(seq, pattern, overlapping=overlapping)
        present = int(count > 0)
        density = match_density(seq, pattern, overlapping=overlapping)
        first_pos = first_match_position(seq, pattern, overlapping=overlapping)
        last_pos = last_match_position(seq, pattern, overlapping=overlapping)
        span = match_span(seq, pattern, overlapping=overlapping)

        out[f"uregex_{motif_name}_present"] = present
        out[f"uregex_{motif_name}_count"] = count
        out[f"uregex_{motif_name}_density"] = density
        out[f"uregex_{motif_name}_first_pos"] = first_pos
        out[f"uregex_{motif_name}_last_pos"] = last_pos
        out[f"uregex_{motif_name}_first_pos_norm"] = normalized_position(first_pos, seq_len)
        out[f"uregex_{motif_name}_last_pos_norm"] = normalized_position(last_pos, seq_len)
        out[f"uregex_{motif_name}_span"] = span
        out[f"uregex_{motif_name}_span_norm"] = normalized_position(span, seq_len) if not np.isnan(span) else np.nan

        if include_terminal:
            out[f"uregex_{motif_name}_nterm_present_w{terminal_window}"] = terminal_presence(
                seq, pattern, side="N", window=terminal_window, overlapping=overlapping
            )
            out[f"uregex_{motif_name}_cterm_present_w{terminal_window}"] = terminal_presence(
                seq, pattern, side="C", window=terminal_window, overlapping=overlapping
            )

    return out


## Functional usage on one sequence

In [6]:

example = regex_motif_descriptors(
    df_demo.loc[0, "sequence"],
    USER_REGEX_PATTERNS,
    overlapping=False,
    include_terminal=True,
    terminal_window=10,
)
list(example.items())[:20]


[('uregex_length', 24),
 ('uregex_valid_residue_count', 24),
 ('uregex_overlapping_mode', 0),
 ('uregex_basic_pair_present', 1),
 ('uregex_basic_pair_count', 1),
 ('uregex_basic_pair_density', 0.041666666666666664),
 ('uregex_basic_pair_first_pos', 23.0),
 ('uregex_basic_pair_last_pos', 23.0),
 ('uregex_basic_pair_first_pos_norm', 0.9583333333333334),
 ('uregex_basic_pair_last_pos_norm', 0.9583333333333334),
 ('uregex_basic_pair_span', nan),
 ('uregex_basic_pair_span_norm', nan),
 ('uregex_basic_pair_nterm_present_w10', 0),
 ('uregex_basic_pair_cterm_present_w10', 1),
 ('uregex_acidic_pair_present', 0),
 ('uregex_acidic_pair_count', 0),
 ('uregex_acidic_pair_density', 0.0),
 ('uregex_acidic_pair_first_pos', nan),
 ('uregex_acidic_pair_last_pos', nan),
 ('uregex_acidic_pair_first_pos_norm', nan)]

## Apply regex motif descriptors to the full dataset

In [7]:

df_uregex = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: regex_motif_descriptors(
                x,
                USER_REGEX_PATTERNS,
                overlapping=False,
                include_terminal=True,
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_uregex.head()


,sequence_id,sequence,label,uregex_length,uregex_valid_residue_count,uregex_overlapping_mode,uregex_basic_pair_present,uregex_basic_pair_count,uregex_basic_pair_density,uregex_basic_pair_first_pos,...,uregex_amide_pair_count,uregex_amide_pair_density,uregex_amide_pair_first_pos,uregex_amide_pair_last_pos,uregex_amide_pair_first_pos_norm,uregex_amide_pair_last_pos_norm,uregex_amide_pair_span,uregex_amide_pair_span_norm,uregex_amide_pair_nterm_present_w10,uregex_amide_pair_cterm_present_w10
0,uregex_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.0,1.0,1.0,0.041667,23.0,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,uregex_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.0,0.0,0.0,0.000000,NaN,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,uregex_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.0,1.0,6.0,0.333333,1.0,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
3,uregex_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.0,0.0,0.0,0.000000,NaN,...,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
4,uregex_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.0,0.0,0.0,0.000000,NaN,...,1.0,0.052632,15.0,15.0,0.789474,0.789474,NaN,NaN,0.0,1.0


## Overlapping vs non-overlapping example

In [8]:

pattern = r"G{2}"
seq = "GGGGGG"

non_overlapping = match_count(seq, pattern, overlapping=False)
overlapping = match_count(seq, pattern, overlapping=True)

print("Non-overlapping count:", non_overlapping)
print("Overlapping count:", overlapping)


Non-overlapping count: 3
Overlapping count: 5


## Inspect descriptor columns

In [9]:

uregex_cols = [c for c in df_uregex.columns if c.startswith("uregex_") and c not in {"uregex_length", "uregex_valid_residue_count", "uregex_overlapping_mode"}]
len(uregex_cols), uregex_cols[:18]


(88,
 ['uregex_basic_pair_present',
  'uregex_basic_pair_count',
  'uregex_basic_pair_density',
  'uregex_basic_pair_first_pos',
  'uregex_basic_pair_last_pos',
  'uregex_basic_pair_first_pos_norm',
  'uregex_basic_pair_last_pos_norm',
  'uregex_basic_pair_span',
  'uregex_basic_pair_span_norm',
  'uregex_basic_pair_nterm_present_w10',
  'uregex_basic_pair_cterm_present_w10',
  'uregex_acidic_pair_present',
  'uregex_acidic_pair_count',
  'uregex_acidic_pair_density',
  'uregex_acidic_pair_first_pos',
  'uregex_acidic_pair_last_pos',
  'uregex_acidic_pair_first_pos_norm',
  'uregex_acidic_pair_last_pos_norm'])

In [10]:

df_uregex[
    [
        "sequence_id",
        "uregex_basic_pair_present",
        "uregex_basic_pair_count",
        "uregex_basic_pair_density",
        "uregex_basic_pair_first_pos_norm",
        "uregex_acidic_pair_present",
        "uregex_gly_run_2plus_present",
        "uregex_ser_thr_patch_cterm_present_w10",
    ]
]


,sequence_id,uregex_basic_pair_present,uregex_basic_pair_count,uregex_basic_pair_density,uregex_basic_pair_first_pos_norm,uregex_acidic_pair_present,uregex_gly_run_2plus_present,uregex_ser_thr_patch_cterm_present_w10
0,uregex_1,1.0,1.0,0.041667,0.958333,0.0,0.0,0.0
1,uregex_2,0.0,0.0,0.000000,NaN,0.0,1.0,0.0
2,uregex_3,1.0,6.0,0.333333,0.055556,1.0,0.0,0.0
3,uregex_4,0.0,0.0,0.000000,NaN,1.0,0.0,0.0
4,uregex_5,0.0,0.0,0.000000,NaN,0.0,0.0,1.0
5,uregex_6,0.0,0.0,0.000000,NaN,0.0,0.0,0.0


## Dataset-level summary

In [11]:

uregex_summary = (
    df_uregex[uregex_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

uregex_summary.head(15)


,descriptor,mean_value
0,uregex_aromatic_pair_last_pos,19.000000
1,uregex_aromatic_pair_first_pos,19.000000
2,uregex_basic_pair_last_pos,17.000000
3,uregex_amide_pair_first_pos,15.000000
4,uregex_amide_pair_last_pos,15.000000
5,uregex_basic_pair_first_pos,12.000000
6,uregex_basic_pair_span,10.000000
7,uregex_acidic_pair_last_pos,10.000000
8,uregex_acidic_pair_first_pos,8.000000
9,uregex_ser_thr_patch_first_pos,6.000000


## Sanity checks

In [12]:

assert "uregex_basic_pair_present" in df_uregex.columns
assert "uregex_basic_pair_count" in df_uregex.columns
assert "uregex_basic_pair_density" in df_uregex.columns
assert "uregex_basic_pair_first_pos_norm" in df_uregex.columns
assert "uregex_gly_run_2plus_span_norm" in df_uregex.columns
assert "uregex_ser_thr_patch_cterm_present_w10" in df_uregex.columns
assert df_uregex["uregex_length"].min() > 0

print(f"Number of user-defined regex motif descriptor columns: {len(uregex_cols)}")
print("User-defined regex motif descriptor checks passed.")


Number of user-defined regex motif descriptor columns: 88
User-defined regex motif descriptor checks passed.


## Class-style implementation closer to the real package

In [13]:

class UserRegexPatternDescriptors:
    """Example class-style user-regex implementation for later migration into Roxy."""

    def __init__(self, pattern_dict, overlapping: bool = False, include_terminal: bool = True, terminal_window: int = 10):
        self.pattern_dict = dict(pattern_dict)
        self.overlapping = bool(overlapping)
        self.include_terminal = bool(include_terminal)
        self.terminal_window = int(terminal_window)

    def transform_sequence(self, seq: str) -> dict:
        return regex_motif_descriptors(
            seq,
            self.pattern_dict,
            overlapping=self.overlapping,
            include_terminal=self.include_terminal,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


uregex_transformer = UserRegexPatternDescriptors(
    pattern_dict=USER_REGEX_PATTERNS,
    overlapping=False,
    include_terminal=True,
    terminal_window=10,
)

uregex_matrix = uregex_transformer.transform(df_demo["sequence"].tolist())
uregex_matrix.head()


,uregex_length,uregex_valid_residue_count,uregex_overlapping_mode,uregex_basic_pair_present,uregex_basic_pair_count,uregex_basic_pair_density,uregex_basic_pair_first_pos,uregex_basic_pair_last_pos,uregex_basic_pair_first_pos_norm,uregex_basic_pair_last_pos_norm,...,uregex_amide_pair_count,uregex_amide_pair_density,uregex_amide_pair_first_pos,uregex_amide_pair_last_pos,uregex_amide_pair_first_pos_norm,uregex_amide_pair_last_pos_norm,uregex_amide_pair_span,uregex_amide_pair_span_norm,uregex_amide_pair_nterm_present_w10,uregex_amide_pair_cterm_present_w10
0,24,24,0,1,1,0.041667,23.0,23.0,0.958333,0.958333,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,15,15,0,0,0,0.000000,NaN,NaN,NaN,NaN,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,18,18,0,1,6,0.333333,1.0,11.0,0.055556,0.611111,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,20,20,0,0,0,0.000000,NaN,NaN,NaN,NaN,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,19,19,0,0,0,0.000000,NaN,NaN,NaN,NaN,...,1,0.052632,15.0,15.0,0.789474,0.789474,NaN,NaN,0,1


## Merge transformer output back to the dataset

In [14]:

df_uregex_class = pd.concat([df_demo, uregex_matrix], axis=1)
df_uregex_class.head()


,sequence_id,sequence,label,uregex_length,uregex_valid_residue_count,uregex_overlapping_mode,uregex_basic_pair_present,uregex_basic_pair_count,uregex_basic_pair_density,uregex_basic_pair_first_pos,...,uregex_amide_pair_count,uregex_amide_pair_density,uregex_amide_pair_first_pos,uregex_amide_pair_last_pos,uregex_amide_pair_first_pos_norm,uregex_amide_pair_last_pos_norm,uregex_amide_pair_span,uregex_amide_pair_span_norm,uregex_amide_pair_nterm_present_w10,uregex_amide_pair_cterm_present_w10
0,uregex_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0,1,1,0.041667,23.0,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,uregex_2,GGGGGGGGGGGGGGG,B,15,15,0,0,0,0.000000,NaN,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,uregex_3,KRRKRRKRRKRRDDDDEE,A,18,18,0,1,6,0.333333,1.0,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,uregex_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0,0,0,0.000000,NaN,...,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,uregex_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0,0,0,0.000000,NaN,...,1,0.052632,15.0,15.0,0.789474,0.789474,NaN,NaN,0,1



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move regex helper logic into `roxy/sequence/patterns.py`
- expose a class such as `UserRegexPatternDescriptors`
- allow configurable:
  - overlapping vs non-overlapping matching
  - terminal windows
  - position-aware summaries
  - count-only vs full descriptor mode
- add tests for:
  - empty sequences
  - invalid regex patterns
  - overlapping motif cases
  - motifs absent from the sequence
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [15]:
# df_uregex.to_csv("demo_user_regex_motif_descriptors.csv", index=False)
